# 03 - Excel Report

This notebook builds `../dashboard/Superstore_Excel_Report.xlsx`.

**Design choice:** the analysis sheets use real Excel formulas (`SUMIFS`, `COUNTIFS`, `AVERAGEIFS`)
that point at the `Data` sheet, instead of pasted numbers. If someone edits the data, the report updates.
Those are the formulas a data analyst is expected to know, and every number can be traced and explained.

Excel does not need to be installed to run this notebook. Open the finished file in Excel and
it calculates on opening.

In [1]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

orders = pd.read_csv("../data/processed/orders_clean.csv", parse_dates=["Order Date", "Order Month"])
print(orders.shape)

(9994, 24)


## Helper columns

An order has several lines, so "number of orders" cannot be a simple row count. I add two helper columns
in Pandas: `Order Count` is 1 only on the **first line of each order**, so `SUMIFS` on it counts distinct
orders. `Returned Order` is the same idea for returned orders. (A formula that counts distinct values
for every row would make a 10,000-row sheet slow.)

In [2]:
orders["Order Count"] = (~orders.duplicated("Order ID")).astype(int)
orders["Returned Order"] = ((orders["Order Count"] == 1) & (orders["Returned"] == "Yes")).astype(int)

data_columns = ["Row ID", "Order ID", "Order Date", "Customer ID", "Segment", "State", "Region", "Category",
                "Sub-Category", "Product Name", "Sales", "Quantity", "Discount", "Profit", "Year",
                "Order Month", "Returned", "Order Count", "Returned Order"]
letters = {name: get_column_letter(i + 1) for i, name in enumerate(data_columns)}
last_row = len(orders) + 1


def rng(name):
    # absolute range on the Data sheet, e.g. Data!$K$2:$K$9995
    return f"Data!${letters[name]}$2:${letters[name]}${last_row}"

## Styling helpers and the Data sheet

In [3]:
HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
MONEY = "#,##0;[Red]-#,##0"
PERCENT = "0.0%;[Red]-0.0%"


def header(ws, row, titles):
    for i, title in enumerate(titles, start=1):
        cell = ws.cell(row=row, column=i, value=title)
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)


def widths(ws, sizes):
    for letter, size in sizes.items():
        ws.column_dimensions[letter].width = size


wb = Workbook()
summary = wb.active
summary.title = "Summary"

data = wb.create_sheet("Data")
data.append(data_columns)
for row in orders[data_columns].itertuples(index=False):
    data.append(list(row))
for r in range(2, last_row + 1):
    data[f"{letters['Order Date']}{r}"].number_format = "yyyy-mm-dd"
    data[f"{letters['Order Month']}{r}"].number_format = "mmm yyyy"
table = Table(displayName="OrdersData", ref=f"A1:{get_column_letter(len(data_columns))}{last_row}")
table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
data.add_table(table)
data.freeze_panes = "A2"
widths(data, {"B": 16, "C": 12, "D": 12, "E": 12, "F": 16, "J": 45, "P": 12})

## Analysis sheets (all formulas)

In [4]:
# ---- By Year ----
years = wb.create_sheet("By Year")
header(years, 1, ["Year", "Revenue", "Profit", "Orders", "Profit Margin", "Avg Order Value", "Revenue Growth"])
for i, year in enumerate(sorted(orders["Year"].unique()), start=2):
    years[f"A{i}"] = int(year)
    years[f"B{i}"] = f"=SUMIFS({rng('Sales')},{rng('Year')},A{i})"
    years[f"C{i}"] = f"=SUMIFS({rng('Profit')},{rng('Year')},A{i})"
    years[f"D{i}"] = f"=SUMIFS({rng('Order Count')},{rng('Year')},A{i})"
    years[f"E{i}"] = f"=C{i}/B{i}"
    years[f"F{i}"] = f"=B{i}/D{i}"
    if i > 2:
        years[f"G{i}"] = f"=B{i}/B{i - 1}-1"
for r in range(2, 6):
    for col in "BCF":
        years[f"{col}{r}"].number_format = MONEY
    years[f"D{r}"].number_format = "#,##0"
    years[f"E{r}"].number_format = PERCENT
    years[f"G{r}"].number_format = PERCENT
widths(years, {"A": 8, "B": 14, "C": 14, "D": 10, "E": 14, "F": 16, "G": 16})

chart = BarChart()
chart.title = "Revenue and profit by year"
chart.add_data(Reference(years, min_col=2, max_col=3, min_row=1, max_row=5), titles_from_data=True)
chart.set_categories(Reference(years, min_col=1, min_row=2, max_row=5))
chart.x_axis.delete = False
chart.y_axis.delete = False
chart.width, chart.height = 18, 8
years.add_chart(chart, "A8")

In [5]:
# ---- By Month (+ seasonality) ----
months = wb.create_sheet("By Month")
header(months, 1, ["Month", "Revenue", "Profit", "Orders", "Month No"])
month_list = sorted(orders["Order Month"].unique())
for i, month in enumerate(month_list, start=2):
    months[f"A{i}"] = pd.Timestamp(month).to_pydatetime()
    months[f"A{i}"].number_format = "mmm yyyy"
    months[f"B{i}"] = f"=SUMIFS({rng('Sales')},{rng('Order Month')},A{i})"
    months[f"C{i}"] = f"=SUMIFS({rng('Profit')},{rng('Order Month')},A{i})"
    months[f"D{i}"] = f"=SUMIFS({rng('Order Count')},{rng('Order Month')},A{i})"
    months[f"E{i}"] = f"=MONTH(A{i})"
    months[f"B{i}"].number_format = MONEY
    months[f"C{i}"].number_format = MONEY
last_month_row = len(month_list) + 1

header_cells = ["Month No", "Month", "Avg Monthly Revenue", "Share of Yearly Revenue"]
for i, title in enumerate(header_cells, start=7):
    cell = months.cell(row=1, column=i, value=title)
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = HEADER_FILL
    cell.alignment = Alignment(horizontal="center", wrap_text=True)
for n in range(1, 13):
    r = n + 1
    months[f"G{r}"] = n
    months[f"H{r}"] = pd.Timestamp(2000, n, 1).strftime("%b")
    months[f"I{r}"] = f"=AVERAGEIFS($B$2:$B${last_month_row},$E$2:$E${last_month_row},G{r})"
    months[f"J{r}"] = f"=I{r}/SUM($I$2:$I$13)"
    months[f"I{r}"].number_format = MONEY
    months[f"J{r}"].number_format = PERCENT
widths(months, {"A": 12, "B": 14, "C": 14, "D": 10, "E": 10, "G": 10, "H": 8, "I": 18, "J": 18})

line = LineChart()
line.title = "Monthly revenue"
line.add_data(Reference(months, min_col=2, min_row=1, max_row=last_month_row), titles_from_data=True)
line.set_categories(Reference(months, min_col=1, min_row=2, max_row=last_month_row))
line.x_axis.number_format = "mmm yy"
line.x_axis.delete = False
line.y_axis.delete = False
line.width, line.height = 22, 8
months.add_chart(line, "L2")

seasonal = BarChart()
seasonal.title = "Average revenue by calendar month"
seasonal.add_data(Reference(months, min_col=9, min_row=1, max_row=13), titles_from_data=True)
seasonal.set_categories(Reference(months, min_col=8, min_row=2, max_row=13))
seasonal.legend = None
seasonal.x_axis.delete = False
seasonal.y_axis.delete = False
seasonal.width, seasonal.height = 22, 8
months.add_chart(seasonal, "L20")

In [6]:
# ---- By Category ----
cats = wb.create_sheet("By Category")
header(cats, 1, ["Category", "Sub-Category", "Revenue", "Profit", "Profit Margin", "Avg Discount"])
sub_order = orders.groupby(["Category", "Sub-Category"])["Profit"].sum().sort_values().reset_index()
for i, row in enumerate(sub_order.itertuples(index=False), start=2):
    cats[f"A{i}"] = row[0]
    cats[f"B{i}"] = row[1]
    cats[f"C{i}"] = f"=SUMIFS({rng('Sales')},{rng('Sub-Category')},B{i})"
    cats[f"D{i}"] = f"=SUMIFS({rng('Profit')},{rng('Sub-Category')},B{i})"
    cats[f"E{i}"] = f"=D{i}/C{i}"
    cats[f"F{i}"] = f"=AVERAGEIFS({rng('Discount')},{rng('Sub-Category')},B{i})"
    cats[f"C{i}"].number_format = MONEY
    cats[f"D{i}"].number_format = MONEY
    cats[f"E{i}"].number_format = PERCENT
    cats[f"F{i}"].number_format = "0.0%"
last_sub_row = len(sub_order) + 1

# category totals under the sub-category table
start = last_sub_row + 3
header(cats, start, ["Category", "", "Revenue", "Profit", "Profit Margin", "Revenue Share"])
for j, category in enumerate(["Technology", "Furniture", "Office Supplies"], start=start + 1):
    cats[f"A{j}"] = category
    cats[f"C{j}"] = f"=SUMIFS({rng('Sales')},{rng('Category')},A{j})"
    cats[f"D{j}"] = f"=SUMIFS({rng('Profit')},{rng('Category')},A{j})"
    cats[f"E{j}"] = f"=D{j}/C{j}"
    cats[f"F{j}"] = f"=C{j}/SUM({rng('Sales')})"
    cats[f"C{j}"].number_format = MONEY
    cats[f"D{j}"].number_format = MONEY
    cats[f"E{j}"].number_format = PERCENT
    cats[f"F{j}"].number_format = "0.0%"
widths(cats, {"A": 18, "B": 16, "C": 14, "D": 14, "E": 14, "F": 14})

bar = BarChart()
bar.type = "bar"
bar.title = "Profit by sub-category"
bar.add_data(Reference(cats, min_col=4, min_row=1, max_row=last_sub_row), titles_from_data=True)
bar.set_categories(Reference(cats, min_col=2, min_row=2, max_row=last_sub_row))
bar.legend = None
bar.series[0].invertIfNegative = False
bar.x_axis.delete = False
bar.y_axis.delete = False
bar.width, bar.height = 18, 10
cats.add_chart(bar, "H2")

In [7]:
# ---- By Region ----
regions = wb.create_sheet("By Region")
header(regions, 1, ["Region", "Revenue", "Profit", "Profit Margin", "Orders", "Avg Discount", "Returned Orders", "Return Rate"])
region_list = orders.groupby("Region")["Sales"].sum().sort_values(ascending=False).index
for i, region in enumerate(region_list, start=2):
    regions[f"A{i}"] = region
    regions[f"B{i}"] = f"=SUMIFS({rng('Sales')},{rng('Region')},A{i})"
    regions[f"C{i}"] = f"=SUMIFS({rng('Profit')},{rng('Region')},A{i})"
    regions[f"D{i}"] = f"=C{i}/B{i}"
    regions[f"E{i}"] = f"=SUMIFS({rng('Order Count')},{rng('Region')},A{i})"
    regions[f"F{i}"] = f"=AVERAGEIFS({rng('Discount')},{rng('Region')},A{i})"
    regions[f"G{i}"] = f"=SUMIFS({rng('Returned Order')},{rng('Region')},A{i})"
    regions[f"H{i}"] = f"=G{i}/E{i}"
    regions[f"B{i}"].number_format = MONEY
    regions[f"C{i}"].number_format = MONEY
    regions[f"D{i}"].number_format = PERCENT
    regions[f"F{i}"].number_format = "0.0%"
    regions[f"H{i}"].number_format = "0.0%"
widths(regions, {"A": 12, "B": 14, "C": 14, "D": 14, "E": 10, "F": 14, "G": 16, "H": 14})

chart = BarChart()
chart.title = "Revenue and profit by region"
chart.add_data(Reference(regions, min_col=2, max_col=3, min_row=1, max_row=5), titles_from_data=True)
chart.set_categories(Reference(regions, min_col=1, min_row=2, max_row=5))
chart.x_axis.delete = False
chart.y_axis.delete = False
chart.width, chart.height = 18, 8
regions.add_chart(chart, "A8")

In [8]:
# ---- Discount Impact ----
disc = wb.create_sheet("Discount Impact")
header(disc, 1, ["Discount Band", "Above", "Up to (incl.)", "Order Lines", "Revenue", "Profit", "Profit Margin"])
bands = [("No discount", -1, 0), ("Up to 20%", 0, 0.2), ("21-40%", 0.2, 0.4), ("Over 40%", 0.4, 1)]
for i, (label, low, high) in enumerate(bands, start=2):
    disc[f"A{i}"] = label
    disc[f"B{i}"] = low
    disc[f"C{i}"] = high
    disc[f"B{i}"].font = Font(color="0000FF")
    disc[f"C{i}"].font = Font(color="0000FF")
    criteria = f'{rng("Discount")},">"&B{i},{rng("Discount")},"<="&C{i}'
    disc[f"D{i}"] = f"=COUNTIFS({criteria})"
    disc[f"E{i}"] = f"=SUMIFS({rng('Sales')},{criteria})"
    disc[f"F{i}"] = f"=SUMIFS({rng('Profit')},{criteria})"
    disc[f"G{i}"] = f"=F{i}/E{i}"
    disc[f"E{i}"].number_format = MONEY
    disc[f"F{i}"].number_format = MONEY
    disc[f"G{i}"].number_format = PERCENT
disc["A7"] = "Blue cells are my own choice of band limits and can be changed. The bands must not overlap."
disc["A7"].font = Font(italic=True)
widths(disc, {"A": 16, "B": 10, "C": 14, "D": 12, "E": 14, "F": 14, "G": 14})

chart = BarChart()
chart.title = "Profit margin by discount band"
chart.add_data(Reference(disc, min_col=7, min_row=1, max_row=5), titles_from_data=True)
chart.set_categories(Reference(disc, min_col=1, min_row=2, max_row=5))
chart.legend = None
chart.series[0].invertIfNegative = False
chart.x_axis.delete = False
chart.y_axis.delete = False
chart.width, chart.height = 16, 8
disc.add_chart(chart, "A9")

In [9]:
# ---- Products ----
products = wb.create_sheet("Products")
top = orders.groupby("Product Name")["Sales"].sum().nlargest(10).index
worst = orders.groupby("Product Name")["Profit"].sum().nsmallest(10).index
assert not any(ch in name for name in list(top) + list(worst) for ch in "*?~")  # wildcard characters would break SUMIFS


def product_table(start_row, title, names):
    products[f"A{start_row}"] = title
    products[f"A{start_row}"].font = Font(bold=True, size=12)
    header(products, start_row + 1, ["Product", "Revenue", "Profit", "Profit Margin"])
    for i, name in enumerate(names, start=start_row + 2):
        products[f"A{i}"] = name
        products[f"B{i}"] = f"=SUMIFS({rng('Sales')},{rng('Product Name')},A{i})"
        products[f"C{i}"] = f"=SUMIFS({rng('Profit')},{rng('Product Name')},A{i})"
        products[f"D{i}"] = f"=C{i}/B{i}"
        products[f"B{i}"].number_format = MONEY
        products[f"C{i}"].number_format = MONEY
        products[f"D{i}"].number_format = PERCENT


product_table(1, "Top 10 products by revenue", top)
product_table(14, "10 products that lose the most money", worst)
widths(products, {"A": 75, "B": 14, "C": 14, "D": 14})

## Summary sheet

The KPI cells are formulas. The findings text comes from the SQL analysis in notebook 02 and the
README explains each one.

In [10]:
summary["A1"] = "Retail Sales & Business Performance Analysis"
summary["A1"].font = Font(bold=True, size=16)
summary["A2"] = "Data: Tableau Sample Superstore (fictional retailer), orders 2015-2018. All KPIs below are formulas on the Data sheet."
summary["A2"].font = Font(italic=True)

kpis = [
    ("Total Revenue", f"=SUM({rng('Sales')})", MONEY),
    ("Total Profit", f"=SUM({rng('Profit')})", MONEY),
    ("Total Orders", f"=SUM({rng('Order Count')})", "#,##0"),
    ("Average Order Value", "=B4/B6", "#,##0.00"),
    ("Profit Margin", "=B5/B4", PERCENT),
    ("Returned Orders", f"=SUM({rng('Returned Order')})", "#,##0"),
    ("Return Rate", "=B9/B6", PERCENT),
]
for i, (label, formula, fmt) in enumerate(kpis, start=4):
    summary[f"A{i}"] = label
    summary[f"A{i}"].font = Font(bold=True)
    summary[f"B{i}"] = formula
    summary[f"B{i}"].number_format = fmt

summary["A13"] = "Key findings (from the SQL analysis)"
summary["A13"].font = Font(bold=True, size=12)
findings = [
    "1. Discounts above 20% turn sales into losses: margin is +29.5% with no discount, +11.9% up to 20%, -15.3% at 21-40% and -77.4% above 40% (see Discount Impact).",
    "2. Furniture earns only 2.5% margin against about 17% for the other two categories. Tables lose money (see By Category).",
    "3. Central has the lowest margin (7.9%) and the highest average discount (24.0%) (see By Region).",
    "4. The West has a return rate of 11.7% against about 3% elsewhere (see By Region).",
    "5. September, November and December bring about 43% of yearly revenue (see By Month).",
]
for i, text in enumerate(findings, start=14):
    summary[f"A{i}"] = text

summary["A20"] = "Notes"
summary["A20"].font = Font(bold=True, size=12)
notes = [
    "Revenue and profit are gross of returns: the data does not say how much of a returned order was refunded.",
    "Order Count and Returned Order on the Data sheet are helper columns (1 on the first line of each order) so that distinct orders can be counted with SUMIFS.",
    "Customer names were removed during cleaning. Sheet formulas refer to the Data sheet, so changing the data updates the report.",
]
for i, text in enumerate(notes, start=21):
    summary[f"A{i}"] = text
widths(summary, {"A": 28, "B": 18})

# use one font for the whole workbook
for ws in wb.worksheets:
    for row in ws.iter_rows():
        for cell in row:
            if cell.value is not None:
                cell.font = Font(name="Arial", size=cell.font.size or 10, bold=cell.font.bold,
                                 italic=cell.font.italic, color=cell.font.color)

wb.calculation.fullCalcOnLoad = True
wb.save("../dashboard/Superstore_Excel_Report.xlsx")
print("saved")

saved
